# TestingAI - Комплексная оценка нейронных сетей для дефектоскопии труб

## Полный цикл оценки моделей:
1. **Загрузка и подготовка данных** из .raw файлов
2. **Обучение модели** с оптимизацией по валидационной выборке
3. **Генерация предсказаний** на тестовой выборке (метки классов и вероятности)
4. **Расчёт базовых метрик**: Accuracy, Precision, Recall, F1-Score, ROC-AUC
5. **Анализ матрицы ошибок** с макро- и взвешенными усреднениями
6. **Статистическая оценка надёжности** (95% доверительные интервалы методом бутстрапа)
7. **Оценка вычислительных характеристик** (время обучения, RAM/VRAM, инференс)
8. **Анализ масштабируемости** и зависимости от размера батча
9. **Анализ кривых обучения** (зависимость от размера выборки)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import struct
import pickle
import json
from datetime import datetime
import time
import warnings
import traceback
from scipy import stats
from typing import Dict, List, Tuple, Optional, Any
import psutil
import os
warnings.filterwarnings('ignore')

## 7. Функции метрик и визуализации

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names=None, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names or ['Class 0', 'Class 1'],
                yticklabels=class_names or ['Class 0', 'Class 1'])
    plt.title('Матрица ошибок (Confusion Matrix)', fontsize=14)
    plt.xlabel('Предсказано', fontsize=12)
    plt.ylabel('Истинное', fontsize=12)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Матрица ошибок сохранена: {save_path}")
    else:
        plt.show()
    plt.close()
    return cm

def plot_roc_curve(y_true, y_probs, save_path=None):
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    auc = roc_auc_score(y_true, y_probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC-кривая', fontsize=14)
    plt.legend(loc="lower right")
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 ROC-кривая сохранена: {save_path}")
    else:
        plt.show()
    plt.close()
    return fpr, tpr, auc

def bootstrap_confidence_interval(y_true, y_pred, metric_func, n_bootstrap=1000, confidence=0.95):
    """Расчёт доверительного интервала методом бутстрапа"""
    np.random.seed(42)
    n_samples = len(y_true)
    bootstrap_scores = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        score = metric_func(y_true[indices], y_pred[indices])
        bootstrap_scores.append(score)
    lower = np.percentile(bootstrap_scores, (1 - confidence) / 2 * 100)
    upper = np.percentile(bootstrap_scores, (1 + confidence) / 2 * 100)
    mean = np.mean(bootstrap_scores)
    return mean, lower, upper

def calculate_all_metrics(y_true, y_pred, y_probs):
    """Расчёт всех базовых метрик с макро- и взвешенными усреднениями"""
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    precision_binary, recall_binary, f1_binary, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0)
    auc_roc = roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.5
    
    metrics = {
        'accuracy': accuracy,
        'precision_binary': precision_binary, 'recall_binary': recall_binary, 'f1_binary': f1_binary,
        'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
        'precision_weighted': precision_weighted, 'recall_weighted': recall_weighted, 'f1_weighted': f1_weighted,
        'auc_roc': auc_roc
    }
    return metrics

## 8. Мониторинг памяти и вычислительных ресурсов

In [ ]:
class ResourceMonitor:
    """Мониторинг RAM и VRAM"""
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.peak_ram = 0
        self.peak_vram = 0
    
    def get_ram_usage(self):
        ram_mb = self.process.memory_info().rss / 1024 / 1024
        self.peak_ram = max(self.peak_ram, ram_mb)
        return ram_mb
    
    def get_vram_usage(self):
        if torch.cuda.is_available():
            vram_mb = torch.cuda.memory_allocated() / 1024 / 1024
            self.peak_vram = max(self.peak_vram, vram_mb)
            return vram_mb
        return 0
    
    def reset_peak(self):
        self.peak_ram = 0
        self.peak_vram = 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    
    def get_peak_stats(self):
        stats = {'peak_ram_mb': self.peak_ram, 'peak_vram_mb': self.peak_vram}
        if torch.cuda.is_available():
            stats['peak_vram_mb'] = torch.cuda.max_memory_allocated() / 1024 / 1024
        return stats

monitor = ResourceMonitor()

## 9. Обучение модели с трекингом ресурсов

In [ ]:
def train_with_tracking(model, train_loader, val_loader, epochs=50, lr=0.001, device='cpu', patience=10):
    """Обучение модели с мониторингом времени и памяти"""
    monitor.reset_peak()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    
    start_time = time.time()
    epoch_times = []
    
    print(f"🚀 Начало обучения на {device}")
    print(f"📊 Параметры модели: {count_parameters(model):,}")
    
    for epoch in range(epochs):
        epoch_start = time.time()
        
        # Train
        model.train()
        total_loss, correct, total = 0, 0, 0
        for data, labels, mask, lengths in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(data, lengths)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        train_loss = total_loss / total
        train_acc = 100. * correct / total
        
        # Validate
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for data, labels, mask, lengths in val_loader:
                data, labels = data.to(device), labels.to(device)
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)
                total_loss += loss.item() * data.size(0)
                total += labels.size(0)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
        val_loss = total_loss / total
        val_acc = 100. * correct / total
        
        scheduler.step()
        
        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {epoch_time:.2f}s")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping на эпохе {epoch+1}")
            break
    
    total_time = time.time() - start_time
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    training_stats = {
        'total_time_sec': total_time,
        'avg_epoch_time_sec': np.mean(epoch_times),
        'epochs_completed': len(history['train_loss']),
        'best_val_acc': best_val_acc,
        'peak_ram_mb': monitor.get_peak_stats()['peak_ram_mb'],
        'peak_vram_mb': monitor.get_peak_stats()['peak_vram_mb'],
        'n_parameters': count_parameters(model)
    }
    
    print(f"\n✅ Обучение завершено!")
    print(f"   Время обучения: {total_time:.2f} сек ({total_time/60:.2f} мин)")
    print(f"   Лучшая точность на валидации: {best_val_acc:.2f}%")
    print(f"   Пиковое потребление RAM: {training_stats['peak_ram_mb']:.2f} MB")
    print(f"   Пиковое потребление VRAM: {training_stats['peak_vram_mb']:.2f} MB")
    
    return model, history, training_stats

## 10. Бенчмаркинг инференса

In [ ]:
def benchmark_inference(model, test_loader, batch_sizes=[1, 8, 16, 32], device='cpu', n_warmup=10, n_runs=100):
    """
    Измерение характеристик инференса для разных размеров батча.
    """
    model.eval()
    results = []
    
    print("="*60)
    print("🔬 БЕНЧМАРКИНГ ИНФЕРЕНСА")
    print("="*60)
    
    for bs in batch_sizes:
        # Create a subset loader with specific batch size
        test_dataset = PipeDataset(
            [test_loader.dataset.data_list[i] for i in range(min(100, len(test_loader.dataset)))],
            [test_loader.dataset.labels[i].item() for i in range(min(100, len(test_loader.dataset)))]
        )
        subset_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, collate_fn=collate_fn)
        
        # Warmup
        for _ in range(n_warmup):
            for data, labels, mask, lengths in subset_loader:
                data = data.to(device)
                with torch.no_grad():
                    _ = model(data, lengths)
        
        # Benchmark
        monitor.reset_peak()
        start_time = time.time()
        inference_count = 0
        
        with torch.no_grad():
            for _ in range(n_runs):
                for data, labels, mask, lengths in subset_loader:
                    data = data.to(device)
                    outputs = model(data, lengths)
                    inference_count += data.size(0)
        
        total_time = time.time() - start_time
        peak_stats = monitor.get_peak_stats()
        
        latency_ms = (total_time / inference_count) * 1000 * n_runs
        throughput = inference_count / total_time
        
        result = {
            'batch_size': bs,
            'total_time_sec': total_time,
            'inference_count': inference_count * n_runs,
            'latency_ms': latency_ms,
            'throughput_samples_per_sec': throughput,
            'peak_ram_mb': peak_stats['peak_ram_mb'],
            'peak_vram_mb': peak_stats['peak_vram_mb']
        }
        results.append(result)
        
        print(f"\nBatch Size: {bs}")
        print(f"   Время {n_runs} проходов: {total_time:.2f} сек")
        print(f"   Задержка (Latency): {latency_ms:.3f} мс/образец")
        print(f"   Пропускная способность: {throughput:.2f} образцов/сек")
        print(f"   Пик RAM: {result['peak_ram_mb']:.2f} MB")
        print(f"   Пик VRAM: {result['peak_vram_mb']:.2f} MB")
    
    return results

def analyze_sequence_length_dependency(model, data_dict, device='cpu'):
    """Анализ зависимости времени инференса от длины последовательности"""
    model.eval()
    
    # Group samples by length
    length_groups = {}
    for i, sample in enumerate(data_dict['X_test']):
        length = sample.shape[0]
        if length not in length_groups:
            length_groups[length] = []
        length_groups[length].append(i)
    
    results = []
    print("\n" + "="*60)
    print("📏 ЗАВИСИМОСТЬ ОТ ДЛИНЫ ПОСЛЕДОВАТЕЛЬНОСТИ")
    print("="*60)
    
    for length in sorted(length_groups.keys()):
        indices = length_groups[length][:10]  # Take up to 10 samples
        samples = [data_dict['X_test'][i] for i in indices]
        labels = [data_dict['y_test'][i] for i in indices]
        
        dataset = PipeDataset(samples, labels)
        loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
        
        start_time = time.time()
        with torch.no_grad():
            for data, lbl, mask, lengths in loader:
                data = data.to(device)
                _ = model(data, lengths)
        elapsed = time.time() - start_time
        
        avg_time = elapsed / len(indices) * 1000  # ms
        results.append({'length': length, 'avg_time_ms': avg_time})
        print(f"Длина {length:4d}: {avg_time:.3f} мс/образец")
    
    return results

## 11. Метрики эффективности и кривые обучения

In [ ]:
def calculate_efficiency_metrics(accuracy, training_stats, inference_results):
    """
    Расчёт метрик эффективности:
    TrainingEfficiency = Accuracy / Training_time
    InferenceEfficiency = Accuracy / Inference_latency
    MemoryEfficiency = Accuracy / Peak_Memory_Usage
    """
    training_time = training_stats.get('total_time_sec', 1)
    inference_latency = inference_results[0]['latency_ms'] if inference_results else 1
    peak_memory = training_stats.get('peak_ram_mb', 1) + training_stats.get('peak_vram_mb', 1)
    
    efficiency = {
        'training_efficiency': accuracy / training_time if training_time > 0 else 0,
        'inference_efficiency': accuracy / inference_latency if inference_latency > 0 else 0,
        'memory_efficiency': accuracy / peak_memory if peak_memory > 0 else 0
    }
    return efficiency

def analyze_learning_curves(data_dict, model_class, device='cpu', 
                            sample_sizes=[100, 200, 400, 800, 1600],
                            n_repeats=3, epochs=30, random_state=42):
    """
    Анализ зависимости метрик от размера обучающей выборки.
    """
    np.random.seed(random_state)
    
    print("="*60)
    print("📈 АНАЛИЗ КРИВЫХ ОБУЧЕНИЯ")
    print("="*60)
    
    all_results = []
    
    for n_samples in sample_sizes:
        print(f"\n🔹 Размер выборки: {n_samples}")
        repeat_results = []
        
        for r in range(n_repeats):
            rs = random_state + r
            np.random.seed(rs)
            
            # Sample training data
            n_train = min(n_samples, len(data_dict['X_train']))
            indices = np.random.choice(len(data_dict['X_train']), size=n_train, replace=False)
            
            X_train_sub = [data_dict['X_train'][i] for i in indices]
            y_train_sub = data_dict['y_train'][indices]
            
            # Create loaders
            train_dataset = PipeDataset(X_train_sub, y_train_sub)
            val_dataset = PipeDataset(data_dict['X_val'], data_dict['y_val'])
            
            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
            val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
            
            # Train model
            model = model_class().to(device)
            _, history, stats = train_with_tracking(
                model, train_loader, val_loader, 
                epochs=epochs, lr=0.001, device=device, patience=10
            )
            
            # Evaluate on test
            test_dataset = PipeDataset(data_dict['X_test'], data_dict['y_test'])
            test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
            
            model.eval()
            all_preds, all_probs, all_labels = [], [], []
            with torch.no_grad():
                for data, labels, mask, lengths in test_loader:
                    data, labels = data.to(device), labels.to(device)
                    outputs = model(data, lengths)
                    probs = torch.softmax(outputs, dim=1)
                    _, predicted = outputs.max(1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_probs.extend(probs[:, 1].cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
            
            metrics = calculate_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs))
            
            repeat_results.append({
                'n_samples': n_samples,
                'random_state': rs,
                'train_acc': history['train_acc'][-1] if history['train_acc'] else 0,
                'val_acc': history['val_acc'][-1] if history['val_acc'] else 0,
                'test_accuracy': metrics['accuracy'],
                'test_precision': metrics['precision_binary'],
                'test_recall': metrics['recall_binary'],
                'test_f1': metrics['f1_binary'],
                'test_auc': metrics['auc_roc'],
                'training_time': stats['total_time_sec'],
                'generalization_gap': history['train_acc'][-1] - metrics['accuracy'] if history['train_acc'] else 0
            })
        
        # Aggregate results
        agg = {
            'n_samples': n_samples,
            'test_accuracy_mean': np.mean([r['test_accuracy'] for r in repeat_results]),
            'test_accuracy_std': np.std([r['test_accuracy'] for r in repeat_results]),
            'test_f1_mean': np.mean([r['test_f1'] for r in repeat_results]),
            'test_f1_std': np.std([r['test_f1'] for r in repeat_results]),
            'training_time_mean': np.mean([r['training_time'] for r in repeat_results]),
            'generalization_gap_mean': np.mean([r['generalization_gap'] for r in repeat_results])
        }
        all_results.append(agg)
        
        print(f"   Accuracy: {agg['test_accuracy_mean']:.4f} ± {agg['test_accuracy_std']:.4f}")
        print(f"   F1-Score: {agg['test_f1_mean']:.4f} ± {agg['test_f1_std']:.4f}")
        print(f"   Время обучения: {agg['training_time_mean']:.2f} сек")
        print(f"   Разрыв обобщения: {agg['generalization_gap_mean']:.4f}")
    
    return all_results

def plot_learning_curve_analysis(results, save_path=None):
    """Построение графиков кривых обучения"""
    df = pd.DataFrame(results)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy vs log(N)
    axes[0, 0].errorbar(np.log(df['n_samples']), df['test_accuracy_mean'], 
                        yerr=df['test_accuracy_std'], capsize=5, marker='o', linestyle='-')
    axes[0, 0].set_xlabel('log(N) - логарифм размера выборки')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Зависимость точности от размера выборки')
    axes[0, 0].grid(True, alpha=0.3)
    
    # F1 vs log(N)
    axes[0, 1].errorbar(np.log(df['n_samples']), df['test_f1_mean'], 
                        yerr=df['test_f1_std'], capsize=5, marker='s', linestyle='--', color='orange')
    axes[0, 1].set_xlabel('log(N) - логарифм размера выборки')
    axes[0, 1].set_ylabel('F1-Score')
    axes[0, 1].set_title('Зависимость F1 от размера выборки')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Training time vs N
    axes[1, 0].plot(df['n_samples'], df['training_time_mean'], marker='^', linestyle='-', color='green')
    axes[1, 0].set_xlabel('Размер выборки (N)')
    axes[1, 0].set_ylabel('Время обучения (сек)')
    axes[1, 0].set_title('Зависимость времени обучения от размера выборки')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Generalization gap vs log(N)
    axes[1, 1].plot(np.log(df['n_samples']), df['generalization_gap_mean'], 
                    marker='d', linestyle='-.', color='red')
    axes[1, 1].set_xlabel('log(N) - логарифм размера выборки')
    axes[1, 1].set_ylabel('Разрыв обобщения (Train Acc - Test Acc)')
    axes[1, 1].set_title('Разрыв обобщения')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 График сохранён: {save_path}")
    else:
        plt.show()
    plt.close()